# FitQuest scan worker (SAM 3D Body)

Runs `backend/worker/server.py` on a Colab GPU and exposes it through a Cloudflare quick tunnel.

Before you start:
1. **Runtime → Change runtime type → T4 GPU.**
2. Click the **key icon** (Secrets) in the left bar and add `HF_TOKEN` (your Hugging Face read token, with access to `facebook/sam-3d-body-vith` granted). Turn on **Notebook access**. Never paste the token into a cell.
3. Optional, for the smoke test: add `FIREBASE_API_KEY` (the `API_KEY` value in `GoogleService-Info.plist`).

Run the cells top to bottom. The tunnel URL changes every time; never commit it.

SAM 3D Body © Meta Platforms, Inc., used under the SAM License.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# If the Fitmax repo is private, upload the backend/worker folder to /content/Fitmax/backend/worker instead.
!git clone --depth 1 https://github.com/facebookresearch/sam-3d-body.git /content/sam-3d-body
!git clone --depth 1 https://github.com/pakxander10/Fitmax.git /content/Fitmax

In [ ]:
# Dependencies from sam-3d-body/INSTALL.md (Colab already has PyTorch), then the worker's own.
!pip install -q pytorch-lightning pyrender opencv-python yacs scikit-image einops timm dill pandas rich hydra-core hydra-submitit-launcher hydra-colorlog pyrootutils webdataset chump networkx==3.2.1 roma joblib seaborn wandb appdirs appnope ffmpeg cython jsonlines pytest xtcocotools loguru optree fvcore black pycocotools tensorboard huggingface_hub
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git@a1ce2f9' --no-build-isolation --no-deps
!pip install -q git+https://github.com/microsoft/MoGe.git
!pip install -q -r /content/Fitmax/backend/worker/requirements.txt

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))

In [ ]:
# Starts the worker. The first start downloads the model, detector and FOV weights: allow several minutes.
import os, subprocess, time, urllib.request

env = {
    **os.environ,
    "PYOPENGL_PLATFORM": "egl",
    "SAM3D_REPO": "/content/sam-3d-body",
    "SAM3D_HF_REPO": "facebook/sam-3d-body-vith",
}
worker_log = open("/content/worker.log", "w")
worker = subprocess.Popen(
    ["python", "-m", "uvicorn", "server:app", "--host", "127.0.0.1", "--port", "8000"],
    cwd="/content/Fitmax/backend/worker", env=env, stdout=worker_log, stderr=subprocess.STDOUT,
)
for _ in range(120):
    try:
        print("Worker is up:", urllib.request.urlopen("http://127.0.0.1:8000/health").read().decode())
        break
    except Exception:
        if worker.poll() is not None:
            break
        time.sleep(5)
else:
    print("Still starting after 10 minutes.")
print(open("/content/worker.log").read()[-3000:])

In [ ]:
# Public HTTPS URL for the app. Requests longer than about 100 s get cut off by the tunnel.
import re

!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared
tunnel_log = open("/content/tunnel.log", "w")
tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"], stdout=tunnel_log, stderr=subprocess.STDOUT)
for _ in range(30):
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if match:
        print("Worker URL:", match.group(0))
        break
    time.sleep(2)
else:
    print(open("/content/tunnel.log").read()[-2000:])

In [ ]:
# Optional smoke test with a real Firebase token. Use a test photo you are happy to upload;
# it is deleted from Colab's disk right after the request.
import json, requests
from google.colab import files

api_key = userdata.get("FIREBASE_API_KEY")
signup = requests.post(f"https://identitytoolkit.googleapis.com/v1/accounts:signUp?key={api_key}", json={"returnSecureToken": True}).json()
if "idToken" not in signup:
    print("Could not get a token (the iOS API key may be restricted to the app). Test from the app instead:", signup.get("error"))
else:
    name, photo = next(iter(files.upload().items()))
    try:
        reply = requests.post(
            "http://127.0.0.1:8000/scan",
            headers={"Authorization": f"Bearer {signup['idToken']}"},
            data={"height_cm": 175, "sex": "male"},
            files={"photo": (name, photo)},
            timeout=120,
        )
        print(reply.status_code, json.dumps(reply.json(), indent=2))
    finally:
        os.remove(name)